In [6]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('/content/case07_lie_factor.csv')
display(df)

,chart_id,subject,value_start,value_end,ink_start,ink_end,ink_unit,ink_dimension,how_the_graphic_encodes_it
0,A,Corporate tax rate,30.0,32.0,4.0,24.0,mm,1,"Bars drawn from a 29.6% baseline, so only the ..."
1,B,Federal health budget (Rs bn),40.0,60.0,1.0,1.5,x scale,2,A coin icon scaled 1.5x in BOTH height and width
2,C,Export earnings ($ bn),25.0,30.0,5.0,6.0,cm,1,Bars from zero: 5.0 cm then 6.0 cm
3,D,Air-pollution index,100.0,150.0,8.0,9.0,cm,1,Bars drawn 8.0 cm then 9.0 cm tall
4,E,City population (millions),2.0,8.0,1.0,4.0,x radius,2,A bubble whose RADIUS was set proportional to ...


In [7]:
def compute_lie_factor(initial_data, final_data, initial_graphic, final_graphic, ink_dimension=1):
    # effect = (final - initial) / initial
    data_effect = (final_data - initial_data) / initial_data

    if ink_dimension == 2:
        # For 2D area-encoded representations where length is scaled linearly but represent area
        # graphic_effect = (final_graphic**2 - initial_graphic**2) / initial_graphic**2 if the measurements are linear dimension
        # Or if the measured ink itself is area, it is (final_graphic - initial_graphic) / initial_graphic.
        # Let's write a flexible function and analyze the data structure first.
        graphic_effect = (final_graphic - initial_graphic) / initial_graphic
    else:
        graphic_effect = (final_graphic - initial_graphic) / initial_graphic

    lie_factor = graphic_effect / data_effect
    return data_effect, graphic_effect, lie_factor

print("Helper function defined. Let's inspect the data columns to construct the precise calculations.")

Helper function defined. Let's inspect the data columns to construct the precise calculations.


In [8]:
import pandas as pd
import numpy as np

# Standard data for the charts
data = {
    'Chart': ['A', 'B', 'C', 'D', 'E'],
    'v_start': [30.0, 40.0, 25.0, 100.0, 2.0],
    'v_end': [32.0, 60.0, 30.0, 150.0, 8.0],
    'ink_start': [4.0, 1.0, 5.0, 8.0, 1.0],
    'ink_end': [24.0, 1.5, 6.0, 9.0, 4.0],
    'dim': [1, 2, 1, 1, 2]
}

results = []
for i in range(5):
    v_s, v_e = data['v_start'][i], data['v_end'][i]
    i_s, i_e = data['ink_start'][i], data['ink_end'][i]
    dim = data['dim'][i]

    data_effect = (v_e - v_s) / v_s
    if dim == 2:
        # For B and E, the measurement (linear scale or radius) scales quadratically in terms of actual ink area used
        graphic_effect = (i_e**2 - i_s**2) / (i_s**2)
    else:
        graphic_effect = (i_e - i_s) / i_s

    lie_factor = graphic_effect / data_effect

    if 0.95 <= abs(lie_factor) <= 1.05:
        verdict = "Honest"
    elif abs(lie_factor) > 1.05:
        verdict = "Overstated (Exaggerated)"
    else:
        verdict = "Understated"

    results.append({
        'Chart': data['Chart'][i],
        'Data Effect': data_effect,
        'Graphic Effect': graphic_effect,
        'Lie Factor': lie_factor,
        'Verdict': verdict
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

Chart  Data Effect  Graphic Effect  Lie Factor                  Verdict
    A     0.066667           5.000       75.00 Overstated (Exaggerated)
    B     0.500000           1.250        2.50 Overstated (Exaggerated)
    C     0.200000           0.200        1.00                   Honest
    D     0.500000           0.125        0.25              Understated
    E     3.000000          15.000        5.00 Overstated (Exaggerated)


In [9]:
try:
    import vizlib
    print("vizlib imported successfully.")
except ImportError:
    print("vizlib not found. We will implement standard lie factor calculation formulas manually.")

vizlib not found. We will implement standard lie factor calculation formulas manually.


In [10]:
# Calculate Lie Factors for each chart A-E according to Tufte's formulation
# If ink_dimension = 2 and linear scaling is used, graphic effect is based on area (i.e., linear measurement squared or scaled factor squared)
# Chart B: "A coin icon scaled 1.5x in BOTH height and width". Thus, ink area increases from 1.0^2 to 1.5^2.
# Chart E: "A bubble whose RADIUS was set proportional to value..." radius goes 1.0 to 4.0, area goes from 1.0^2 to 4.0^2.

results = []
for idx, row in df.iterrows():
    v_start, v_end = row['value_start'], row['value_end']
    ink_start, ink_end = row['ink_start'], row['ink_end']
    dim = row['ink_dimension']

    data_effect = (v_end - v_start) / v_start

    if dim == 2:
        # Since these are linear measures of a 2D symbol, area scales quadratically
        graphic_effect = (ink_end**2 - ink_start**2) / (ink_start**2)
    else:
        graphic_effect = (ink_end - ink_start) / ink_start

    lie_factor = graphic_effect / data_effect

    # Determine verdict
    if 0.95 <= abs(lie_factor) <= 1.05:
        verdict = "Honest"
    elif abs(lie_factor) > 1.05:
        verdict = "Exaggerated / Overstated"
    else:
        verdict = "Understated"

    results.append({
        "Chart": row['chart_id'],
        "Subject": row['subject'],
        "Data Effect": data_effect,
        "Graphic Effect": graphic_effect,
        "Lie Factor": lie_factor,
        "Verdict": verdict
    })

res_df = pd.DataFrame(results)
display(res_df)

,Chart,Subject,Data Effect,Graphic Effect,Lie Factor,Verdict
0,A,Corporate tax rate,0.066667,5.000,75.00,Exaggerated / Overstated
1,B,Federal health budget (Rs bn),0.500000,1.250,2.50,Exaggerated / Overstated
2,C,Export earnings ($ bn),0.200000,0.200,1.00,Honest
3,D,Air-pollution index,0.500000,0.125,0.25,Understated
4,E,City population (millions),3.000000,15.000,5.00,Exaggerated / Overstated
